# P16D cube + NDVI + TempCNN (Rondonia, `samples_deforestation_rondonia_1M.rds`)

Training samples must share the **same timeline** as the prediction cube after `cube_regularize`. If you see **`.check_samples_tile_match_timeline`**, the period/resolution (or dates) do not match how the RDS was built. This notebook follows [`inst/demo-paper-2025/tempcnn_model_training.R`](../demo-paper-2025/tempcnn_model_training.R): **P16D**, **600 m**, bands including cloud + **NDVI** on B04/B08.

Training RDS is loaded by the **worker** from HTTPS (no local `readRDS`). Requires a Python client with ML helpers (e.g. [PondiB/openeo-python-client](https://github.com/PondiB/openeo-python-client)). Adjust credentials for your environment.

## 1. Connect

In [ ]:
import openeo

In [ ]:
connection = openeo.connect("http://127.0.0.1:8000")
connection.authenticate_basic("user", "password")

## 2. List collections and processes

In [ ]:
print("Available collections:", connection.list_collection_ids())
print("\nAvailable processes:", [p["id"] for p in connection.list_processes()])

## 3. Area of interest and time range

In [ ]:
bbox = {
    "west": -63.33,
    "south": -12.03,
    "east": -62.43,
    "north": -11.13,
    "crs": 4326,
}
temporal_extent = ["2022-01-01", "2022-12-31"]

## 4. Load Sentinel-2 collection

In [ ]:
datacube = connection.load_collection(
    "mpc-sentinel-2-l2a",
    spatial_extent=bbox,
    temporal_extent=temporal_extent
)

## 5. Regularize (`P16D`, 300 m )

In [ ]:
datacube = datacube.process(
    process_id="cube_regularize",
    arguments={
        "data": datacube,
        "period": "P16D",
        "resolution": 300,
    },
)

## 6. NDVI

In [ ]:
datacube = datacube.ndvi(red="B04", nir="B08", target_band="NDVI")

## 7. Training samples (URL — worker downloads RDS)

In [ ]:
training_set = "https://github.com/Open-Earth-Monitor/openeocraft/raw/main/inst/demo-paper-2025/data/samples_deforestation_rondonia.rds"

## 8. TempCNN model definition

In [ ]:
tempcnn_model_init = connection.mlm_class_tempcnn(
    optimizer="adam",
    learning_rate=0.0005,
    seed=42,
)

## 9. Train (`ml_fit`)

In [ ]:
tempcnn_model = tempcnn_model_init.fit(
    training_set=training_set,
    target="label",
)

## 10. Predict (`ml_predict`)

On **OpenEOcraft**, `ml_predict` runs `sits_classify` **and** `sits_label_classification`, so the cube is already **hard-classified**. Do **not** chain `ml_label_class` afterward—`ml_label_class` expects a **probability** cube only and will error with *input should be a cube with probabilities*.

If you need probabilities first, use `ml_predict_probabilities` and then `ml_label_class` (see process list).


In [ ]:
datacube = tempcnn_model.predict(datacube)

## 11. Save trained model (`save_ml_model`)

The openEO ML extension defines **`save_ml_model`** for persisting a fitted model. A batch graph has only one `result` node: this call builds the graph whose result is **`save_ml_model`** (train → save). To run it, submit a job from the returned object (see [`02_ml_training.ipynb`](02_ml_training.ipynb)). The GeoTIFF job in section 12 uses a separate graph rooted at **`save_result`**.

In [ ]:
tempcnn_model.save_ml_model(name="tempcnn_rondonia")

## 12. Submit job and download results

Submit the **`save_result`** graph for GeoTIFF output. With multiple tiles, OpenEOcraft mosaics to **`openeocraft_merged.tif`** by default (see process `options` for `merge_tiles` / `target_crs`).

In [ ]:
result = datacube.save_result(format="GeoTiff")

In [ ]:
result

In [ ]:
job = result.create_job(
    title="Deforestation Prediction in Rondonia",
    description="Using TempCNN model to predict deforestation in Rondonia",
)

In [ ]:
job.start_and_wait()
results = job.get_results()
results.download_files("data/output_month")